# INIT

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

# Read FROM BRONZE Layer

In [0]:
df = spark.table('workspace.bronze.crm_cust_info')

# Data Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .otherwise('Unknown')
    )
    .withColumn(
        'cst_marital_status',
        F.when(F.upper(F.col('cst_marital_status')) == 'S', 'Single')
         .when(F.upper(F.col('cst_marital_status')) == 'M', 'Married')
         .otherwise('Unknown')
    )
)

### Renaming the columns

In [0]:
rename_map = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'created_date'
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

## Exploring data

In [0]:
df.limit(10).display()

# Write INTO SILVER Layer

In [0]:
(
    df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.crm_cust_info')
 )